### Import Library

In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
from scipy.optimize import brentq
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

pio.renderers.default = 'colab'


Data pull (yfinance), math (numpy/scipy), and plotting (plotly) — everything the notebook needs.

### Import Data

In [ ]:
ticker = yf.Ticker("SPY")
S = ticker.history(period="1d")['Close'].iloc[-1]
expiries = ticker.options[:6]

chains = []
for exp in expiries:
    calls = ticker.option_chain(exp).calls
    calls['expirationDate'] = exp
    calls['optionType'] = 'call'
    chains.append(calls)

options_chain = pd.concat(chains, ignore_index=True)


Grabs spot price and the first 6 expiries' call chains, tagging each row `optionType = 'call'` (needed later since only calls are pulled). Capped at 6 — far-dated expiries are thin on volume and would just add noise.

### Describe Data

In [ ]:
print(f"Spot: {S:.2f}")
print(f"Contracts pulled: {len(options_chain)}")
display(options_chain.head())
options_chain.info()


Sanity check — confirms the pull worked and shows what columns/dtypes you're working with before touching anything.

### Data Visualization

In [ ]:
hist = ticker.history(period="1y")

fig = go.Figure(go.Scatter(x=hist.index, y=hist['Close'], mode='lines', name='SPY'))
fig.update_layout(title="SPY - 1Y", xaxis_title="Date", yaxis_title="Price")
fig.show()


Quick look at the underlying's 1-year trend — context for whether current IVs look elevated relative to recent price action.

### Data Preprocessing

In [ ]:
df = options_chain[options_chain['volume'] > 0].copy()

df['expirationDate'] = pd.to_datetime(df['expirationDate'])
today = pd.Timestamp.today()
df['T'] = (df['expirationDate'] - today).dt.days / 365.25
df = df[df['T'] > 0]

df['Moneyness'] = df['strike'] / S
df = df[df['Moneyness'].between(0.85, 1.15)]

r = 0.043
q = 0.013
df['r'] = r
df['q'] = q


Drops zero-volume (stale) quotes, computes time-to-maturity in years, drops expired contracts, and restricts to strikes within 15% of spot where liquidity and signal are best. `r`/`q` are flat constants, added as columns so `implied_vol` can read them per row.

### Define Target Variable (y) and Feature Variables (X)

In [ ]:
# X: spot, strike, T, r, q
# y: implied vol, backed out per-contract by inverting the CRR tree price against the mid price


Frames the problem: everything except IV is an input; IV is what gets solved for.

### Train Test Split

In [ ]:
# N/A here — the CRR tree prices directly from parameters, it isn't fit to data.
# IV comes from root-finding on the pricing equation, not a train/test split.


No split needed — this isn't a fitted model, it's an inversion of a known formula per contract.

### Modeling

In [ ]:
def crr_price(S, K, T, r, q, sigma, option_type='call', N=120):
    """American option price via a Cox-Ross-Rubinstein binomial tree."""
    if sigma <= 0 or T <= 0:
        return max(S - K, 0.0) if option_type == 'call' else max(K - S, 0.0)

    dt = T / N
    u = np.exp(sigma * np.sqrt(dt))
    d = 1.0 / u
    disc = np.exp(-r * dt)
    p = (np.exp((r - q) * dt) - d) / (u - d)
    p = min(max(p, 1e-8), 1 - 1e-8)

    j = np.arange(N + 1)
    ST = S * u ** (N - j) * d ** j
    values = np.maximum(ST - K, 0.0) if option_type == 'call' else np.maximum(K - ST, 0.0)

    for i in range(N - 1, -1, -1):
        values = disc * (p * values[:-1] + (1 - p) * values[1:])
        j = np.arange(i + 1)
        Si = S * u ** (i - j) * d ** j
        intrinsic = np.maximum(Si - K, 0.0) if option_type == 'call' else np.maximum(K - Si, 0.0)
        values = np.maximum(values, intrinsic)

    return values[0]


def implied_vol(row):
    K, T, r, q = row['strike'], row['T'], row['r'], row['q']
    opt = row['optionType']
    mid = (row['bid'] + row['ask']) / 2 if row['bid'] > 0 and row['ask'] > 0 else row['lastPrice']

    lo = max(S - K, 0.0) if opt == 'call' else max(K - S, 0.0)
    hi = S if opt == 'call' else K
    if not (lo < mid < hi):
        return np.nan

    try:
        return brentq(lambda sig: crr_price(S, K, T, r, q, sig, opt) - mid,
                       1e-4, 3.0, maxiter=100, xtol=1e-6)
    except (ValueError, RuntimeError):
        return np.nan


`crr_price` prices an American option (call or put) via a Cox-Ross-Rubinstein binomial tree, checking for early exercise at every step. `implied_vol` uses mid-price (steadier than last trade) and American no-arbitrage bounds, then brentq-inverts the tree price to solve for the vol that reproduces the market price.

In [ ]:
df['IV'] = df.apply(implied_vol, axis=1)
df = df.dropna(subset=['IV'])

counts = df['T'].value_counts()
liquid = counts[counts > 4]
front_T = liquid.index.min() if len(liquid) else counts.index[0]
front_month = df[df['T'] == front_T].sort_values('Moneyness')

fig = make_subplots(rows=1, cols=2, specs=[[{"type": "xy"}, {"type": "scene"}]],
                     subplot_titles=("Front-Month Smile", "Vol Surface"))

fig.add_trace(go.Scatter(x=front_month['Moneyness'], y=front_month['IV'], mode='lines+markers',
                          marker=dict(color=front_month['IV'], colorscale='Viridis', size=8)),
              row=1, col=1)
fig.add_vline(x=1.0, line_dash="dash", line_color="red", row=1, col=1, annotation_text="ATM")

fig.add_trace(go.Mesh3d(x=df['Moneyness'], y=df['T'], z=df['IV'], intensity=df['IV'],
                         colorscale='Viridis', opacity=0.9), row=1, col=2)

fig.update_layout(title=f"SPY Vol Surface | Spot {S:.2f}", height=650,
                   margin=dict(l=20, r=20, b=20, t=60), showlegend=False, hovermode="x unified")
fig.update_xaxes(title_text="Moneyness", range=[0.85, 1.15],
                  rangeslider=dict(visible=True, thickness=0.1), row=1, col=1)
fig.update_yaxes(title_text="IV", row=1, col=1)
fig.layout.scene.update(xaxis_title="Moneyness", yaxis_title="T (yrs)", zaxis_title="IV")

fig.show(config={'scrollZoom': True, 'displayModeBar': True, 'modeBarButtonsToRemove': ['lasso2d', 'select2d']})
fig.write_html("spy_iv_surface.html")


Runs the solver on every row, then plots two views: the front-month smile (skew across strikes at one maturity) and the full 3D surface (skew across both strike and maturity).